# Build a small drive station

**Lesson 1 of 3 | Beginner | About 50 minutes**

## Learning scenario

A training workshop is building a small mixer. The mixer needs an
electric motor and a power supply. During commissioning, equipment
names and operating values may change, but other tools must still
understand what each object *is* and how the objects are connected.

We will begin with concrete equipment only. Classes appear later,
after the model itself shows us why they are useful.


## 1. Create a document and an InstanceHierarchy

A `CAEXFile` is the AutomationML document. An
`InstanceHierarchy` holds the equipment occurrences in one system.
We name ours `TrainingWorkshop`.


In [ ]:
from automationml import caex_file, instance_hierarchy

document = caex_file("training-drive-station.aml")
workshop = instance_hierarchy("TrainingWorkshop")
document.instance_hierarchies.append(workshop)


## 2. Add the motor

An `InternalElement` represents an object occurrence. Its `Name` is
readable for people. Its `ID` gives this occurrence a stable
identity inside the document.


In [ ]:
from automationml import internal_element

motor = internal_element("Motor 1", id="motor-1")
workshop.internal_elements.append(motor)


## 3. Describe this particular motor

Attributes hold information about an object. At commissioning time
we know that this motor runs at 1500 rpm and expects 48 V.


In [ ]:
from automationml import attribute

motor.attributes.append(
    attribute("ratedSpeed", 1500, unit="rpm", data_type="xs:double")
)
motor.attributes.append(
    attribute("ratedVoltage", 48, unit="V", data_type="xs:double")
)


### First inspection

We now have enough information to make inspecting the element
worthwhile. Read the output as an object, not as an exchange format:
one identity, one human name, and two properties.


In [ ]:
print(f"{motor.name}  [ID: {motor.id}]")
for item in motor.attributes:
    print(f"  {item.name}: {item.value} {item.unit or ''}".rstrip())


## 4. The motor needs a power supply

The supply is another independent equipment occurrence. Its output
voltage matches the motor, and its rated power is 500 W.


In [ ]:
power_supply = internal_element("Power Supply 1", id="power-supply-1")
workshop.internal_elements.append(power_supply)


In [ ]:
power_supply.attributes.append(
    attribute("outputVoltage", 48, unit="V", data_type="xs:double")
)
power_supply.attributes.append(
    attribute("ratedPower", 500, unit="W", data_type="xs:double")
)


## 5. Give both elements a connection point

An `ExternalInterface` is a connection point on an element. The
motor receives power through `PowerIn`; the supply provides it
through `PowerOut`.


In [ ]:
from automationml import external_interface

motor_power = external_interface("PowerIn", id="motor-1-power-in")
motor.external_interfaces.append(motor_power)


In [ ]:
supply_power = external_interface(
    "PowerOut", id="power-supply-1-power-out"
)
power_supply.external_interfaces.append(supply_power)


## 6. Put the equipment under one connection owner

A CAEX `InternalLink` belongs to the element that contains both
partners. Therefore the motor and supply become children of a
common `Drive Station 1` element. This is our first actual equipment
hierarchy.


In [ ]:
drive_station = internal_element(
    "Drive Station 1",
    id="drive-station-1",
    children=[motor, power_supply],
)
workshop.internal_elements = [drive_station]


The partner syntax below combines a stable element ID with the name
of one of its interfaces. For this learning scenario, read Partner A
as the source and Partner B as the target.


In [ ]:
from automationml import InternalLink

power_link = InternalLink(
    name="MotorPower",
    ref_partner_side_a="power-supply-1:PowerOut",
    ref_partner_side_b="motor-1:PowerIn",
)
drive_station.internal_links.append(power_link)


### Inspect the connected station

The tree shows containment. The final line shows a relationship
between interfaces; that relationship is not the same thing as
parent-child containment.


In [ ]:
print(drive_station.name)
for child in drive_station.internal_elements:
    interfaces = ", ".join(item.name for item in child.external_interfaces)
    print(f"  +-- {child.name}  [{interfaces}]")
print(
    f"  link: {power_link.ref_partner_side_a}"
    f" -> {power_link.ref_partner_side_b}"
)


## 7. A valid structure can still have weak meaning

At this point the validator can confirm that the link partners
exist. But `Motor 1` is only a name we chose. Nothing in the model
formally says that the element is an electric motor rather than a
label that happens to contain the word "Motor".

This distinction matters: **validation can confirm claims that the
model makes, but it cannot invent semantic claims that are absent.**


In [ ]:
issues_without_classes = document.caex_validation_issues(strict_xsd=True)

print("Validation issues:", len(issues_without_classes))
assert issues_without_classes == []


Zero issues means the claims currently present are consistent. It
does **not** mean that the document already carries strong type
semantics.

## 8. Commissioning changes human-facing data

The workshop decides on clearer names, and testing produces a new
motor speed. The IDs and link do not change when element names
change.


In [ ]:
motor.name = "Mixer Drive Motor"
motor.attributes[0].value = "1800"
power_supply.name = "Control Cabinet Power Supply"

print(motor.name, "at", motor.attributes[0].value, "rpm")
print(power_supply.name)
print(power_link.ref_partner_side_a, "->", power_link.ref_partner_side_b)


Names and values should be editable. The stable meaning should not
depend on the current display name. Now we have a concrete reason
to introduce classes.

## 9. Classify the interface meaning

`ElectricalPower` describes the kind of connection independently
from the local names `PowerIn` and `PowerOut`.


In [ ]:
from automationml import InterfaceClassLib, InterfaceFamily

connection_types = InterfaceClassLib(
    name="ConnectionTypes",
    interface_classes=[InterfaceFamily(name="ElectricalPower")],
)
document.interface_class_libs.append(connection_types)


In [ ]:
POWER_INTERFACE_PATH = "ConnectionTypes/ElectricalPower"

motor_power.ref_base_class_path = POWER_INTERFACE_PATH
supply_power.ref_base_class_path = POWER_INTERFACE_PATH


## 10. Classify important attribute meaning

Attribute types give shared meaning to values even when different
classes use local attribute names such as `ratedVoltage` and
`outputVoltage`.


In [ ]:
from automationml import AttributeType, AttributeTypeLib

value_types = AttributeTypeLib(name="ValueTypes")
document.attribute_type_libs.append(value_types)


First, define the meaning and unit of rotational speed.


In [ ]:
speed_type = AttributeType(
    name="RotationalSpeed",
    unit="rpm",
    attribute_data_type="xs:double",
)
value_types.attribute_types.append(speed_type)


Voltage is a second shared meaning used by both pieces of equipment.


In [ ]:
voltage_type = AttributeType(
    name="Voltage",
    unit="V",
    attribute_data_type="xs:double",
)
value_types.attribute_types.append(voltage_type)


In [ ]:
motor.attributes[0].ref_attribute_type = "ValueTypes/RotationalSpeed"
motor.attributes[1].ref_attribute_type = "ValueTypes/Voltage"
power_supply.attributes[0].ref_attribute_type = "ValueTypes/Voltage"


## 11. Define the motor class

A `SystemUnitClass` describes reusable equipment shape. The class
below says what attributes and interfaces an electric motor is
expected to materialize.


In [ ]:
from automationml import SystemUnitFamily

motor_class = SystemUnitFamily(name="ElectricMotor")


Add the expected motor attributes one at a time. These define shape,
so they do not need occurrence-specific values.


In [ ]:
motor_class.attributes.append(
    attribute(
        "ratedSpeed", unit="rpm", data_type="xs:double",
        ref_attribute_type="ValueTypes/RotationalSpeed",
    )
)


In [ ]:
motor_class.attributes.append(
    attribute(
        "ratedVoltage", unit="V", data_type="xs:double",
        ref_attribute_type="ValueTypes/Voltage",
    )
)


Finally, make `PowerIn` part of the expected motor interface set.


In [ ]:
motor_class.external_interfaces.append(
    external_interface(
        "PowerIn", id="class-motor-power-in",
        ref_base_class_path=POWER_INTERFACE_PATH,
    )
)


## 12. Define the power-supply class

The same pattern gives the supply a stable type. Its two attributes
and power output become the expected class shape.


In [ ]:
supply_class = SystemUnitFamily(name="PowerSupply")


In [ ]:
supply_class.attributes.append(
    attribute(
        "outputVoltage", unit="V", data_type="xs:double",
        ref_attribute_type="ValueTypes/Voltage",
    )
)


In [ ]:
supply_class.attributes.append(
    attribute("ratedPower", unit="W", data_type="xs:double")
)


In [ ]:
supply_class.external_interfaces.append(
    external_interface(
        "PowerOut", id="class-supply-power-out",
        ref_base_class_path=POWER_INTERFACE_PATH,
    )
)


## 13. Put the classes in a library and reference them

Library name plus class name forms a stable AML path. Instance names
may continue to change without changing that type reference.


In [ ]:
from automationml import SystemUnitClassLib

equipment_types = SystemUnitClassLib(
    name="EquipmentTypes",
    system_unit_classes=[motor_class, supply_class],
)
document.system_unit_class_libs.append(equipment_types)


In [ ]:
motor.ref_base_system_unit_path = "EquipmentTypes/ElectricMotor"
power_supply.ref_base_system_unit_path = "EquipmentTypes/PowerSupply"

print(motor.name, "is a", motor.ref_base_system_unit_path)
print(power_supply.name, "is a", power_supply.ref_base_system_unit_path)


## 14. Validate the semantic claims

The validator can now resolve the type paths and compare each
instance with its class shape. That is much stronger than checking
only whether the object tree is internally consistent.


In [ ]:
issues = document.caex_validation_issues(strict_xsd=True)

for issue in issues:
    print(issue.severity, issue.code, issue.path)

assert issues == []
print("The classified motor and supply validate without issues.")


## 15. Your turn: classify the containing station

`Drive Station 1` is also an InternalElement, but it does not yet
reference a SystemUnitClass. Complete the hierarchy by:

1. creating a `SystemUnitFamily` named `DriveStation`;
2. adding it to `equipment_types.system_unit_classes`;
3. setting the station's path to `EquipmentTypes/DriveStation`.

Use the empty cell below. Keep the existing motor and supply classes.


In [ ]:
# Your code here:


In [ ]:
if drive_station.ref_base_system_unit_path is None:
    print("Challenge ready: Drive Station 1 still needs its class.")
else:
    target = document.reference_index().resolve_system_unit_class(
        drive_station.ref_base_system_unit_path
    )
    assert target is not None
    assert document.caex_validation_issues(strict_xsd=True) == []
    print("Complete:", drive_station.name, "is a", target.aml_path)


<details>
<summary><strong>Reveal one solution</strong></summary>

```python
drive_station_class = SystemUnitFamily(name="DriveStation")
equipment_types.system_unit_classes.append(drive_station_class)
drive_station.ref_base_system_unit_path = "EquipmentTypes/DriveStation"
```

Run the check cell again after adding the solution.
</details>

## What you built

- an InstanceHierarchy containing a small equipment hierarchy;
- instance attributes and external interfaces;
- an InternalLink between two interface partners;
- interface, attribute, and equipment semantics that survive renaming;
- a final class-design task for the containing element.

Lesson 2 will take a completed AutomationML model and focus only on
JSON and XML exchange.
